# 03 - Outputs: Evaluation, Deployment, Inspection

Open the artifacts produced by `02_training_smoke_run.ipynb` (or by
`scripts/train_small_vector.py`) and summarize them.

Set `HABCONN_RUN_DIR` to point this notebook at a different run.

In [13]:
import os
from pathlib import Path
pkg_root = Path.cwd().resolve()
if pkg_root.name != 'habconn':
    pkg_root = Path.cwd().parent
default_run = pkg_root / 'tmp' / 'notebooks' / 'experiments' / 'notebook_smoke'
fallback_run = pkg_root / 'tmp' / 'experiments' / 'baseline_small_vector_001'
env_run = os.environ.get('HABCONN_RUN_DIR')
if env_run:
    run_dir = Path(env_run)
elif default_run.exists():
    run_dir = default_run
else:
    run_dir = fallback_run
print('run_dir:', run_dir)
assert run_dir.exists(), f'run directory not found: {run_dir}'

run_dir: C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke


## Config + summary

In [14]:
import json
config = json.loads((run_dir / 'config.json').read_text())
summary = json.loads((run_dir / 'baseline_summary.json').read_text())
print('run_name           :', config['run_name'])
print('seed               :', config['seed'])
print('budget / k         :', config['budget'], '/', config['k'])
print('total_timesteps    :', config['total_timesteps'])
print('n_envs             :', config['n_envs'])
ev = summary['evaluation']
print('eval mean return   :', f"{ev['mean_return']:.3e}")
print('eval mean final PC :', f"{ev['mean_final_pc']:.6e}")
print('eval mean delta PC :', f"{ev['mean_delta_pc']:.3e}")

run_name           : notebook_smoke
seed               : 42
budget / k         : 3 / 10
total_timesteps    : 50
n_envs             : 1
eval mean return   : 4.792e-06
eval mean final PC : 2.652311e-05
eval mean delta PC : 4.792e-06


## Evaluation comparison

In [15]:
import json
comp = json.loads((run_dir / 'evaluation' / 'comparison.json').read_text())
print('methods :', list(comp['method_means']))
for method, m in comp['method_means'].items():
    print(f'  {method:14s} mean_final_pc={m["mean_final_pc"]:.6e}  mean_delta_pc={m["mean_delta_pc"]:.3e}')

methods : ['trained_policy', 'random_valid', 'lowest_cost', 'largest_area']
  trained_policy mean_final_pc=2.652311e-05  mean_delta_pc=4.792e-06
  random_valid   mean_final_pc=2.546508e-05  mean_delta_pc=3.734e-06
  lowest_cost    mean_final_pc=2.583473e-05  mean_delta_pc=4.104e-06
  largest_area   mean_final_pc=2.612015e-05  mean_delta_pc=4.389e-06


## Model selection

In [16]:
import json
sel = json.loads((run_dir / 'selection' / 'model_selection.json').read_text())
print('selection_metric  :', sel['selection_metric'])
print('selection_mode    :', sel['selection_mode'])
print('tie_break_rule    :', sel.get('tie_break_rule'))
print('selected_id       :', sel['selected_candidate_id'])
print('selected_type     :', sel['selected_candidate_type'])
print('selected_timestep :', sel['selected_candidate_timestep'])

selection_metric  : mean_final_pc
selection_mode    : max
tie_break_rule    : Among candidates of equal metric value, prefer the larger timestep; if timesteps also tie, prefer 'final' over 'checkpoint'.
selected_id       : final_model
selected_type     : final
selected_timestep : 56


## Deployment

In [17]:
import json
dep = json.loads((run_dir / 'deployment' / 'deployment_summary.json').read_text())
print('model_path       :', dep['model_path'])
print('selected_pu_ids  :', dep['selected_pu_ids'])
print('final_pc         :', f"{dep['final_pc']:.6e}")
print('delta_pc_total   :', f"{dep['delta_pc_total']:.3e}")

model_path       : C:\Users\dev\work\tum\drl-sp\08_pkg\habconn\tmp\notebooks\experiments\notebook_smoke\models\best_model.zip
selected_pu_ids  : [3, 4, 5]
final_pc         : 2.652311e-05
delta_pc_total   : 4.792e-06


## Selected planning units (CSV)

In [18]:
import pandas as pd
df = pd.read_csv(run_dir / 'deployment' / 'selected_planning_units.csv')
print(df.shape)
df

(3, 4)


,pu_id,lyr_1,cost,selection_order
0,3,1878,1.0,0
1,4,1879,1.0,1
2,5,1886,1.0,2


## Observation schema (consumed vs unused)

In [19]:
import json
schema = json.loads((run_dir / 'inspection' / 'observation_schema.json').read_text())
for info in schema['keys']:
    key = info['name']
    flag = 'used  ' if info['consumed_by_flat_extractor'] else 'unused'
    print(f'{key:22s} {flag}  shape={tuple(info["shape"])}  dtype={info["dtype"]}')

action_mask            used    shape=(10,)  dtype=bool
candidate_areas        used    shape=(10,)  dtype=float32
candidate_costs        used    shape=(10,)  dtype=float32
candidate_ids          unused  shape=(10,)  dtype=int32
eligibility_mask       unused  shape=(79,)  dtype=bool
node_areas             unused  shape=(79,)  dtype=float32
node_costs             unused  shape=(79,)  dtype=float32
node_mask              unused  shape=(79,)  dtype=bool
selected_mask          unused  shape=(79,)  dtype=bool
budget_fraction        used    shape=(1,)  dtype=float32
current_pc             used    shape=(1,)  dtype=float32
remaining_budget       used    shape=(1,)  dtype=float32
selected_fraction      used    shape=(1,)  dtype=float32
step_count             used    shape=(1,)  dtype=int32


## Action trace (one row per step+slot)

In [20]:
trace = pd.read_csv(run_dir / 'inspection' / 'deployment_action_trace.csv')
print(trace.shape)
trace.head(20)

(30, 11)


,step,slot,pu_id,valid,chosen,candidate_cost,candidate_area,remaining_budget_before,current_pc_before,reward_after,pc_after
0,1,0,1,True,False,1.0,9100.0,3.0,0.000022,0.000002,0.000023
1,1,1,2,True,False,1.0,9800.0,3.0,0.000022,0.000002,0.000023
2,1,2,3,True,True,1.0,10000.0,3.0,0.000022,0.000002,0.000023
3,1,3,4,True,False,1.0,9600.0,3.0,0.000022,0.000002,0.000023
4,1,4,5,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
5,1,5,6,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
6,1,6,7,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
7,1,7,8,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
8,1,8,9,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
9,1,9,10,True,False,1.0,10000.0,3.0,0.000022,0.000002,0.000023
